# Track 2 · Stage 4 — Evaluation

## Run **GATE 3 first**, before any generation or training.

`whisper-large-v2` zero-shot on the test set should score **≈ 52.0 MER / 42.9 CBA-HE**.
Inference only — no training, ~3 GB, ~40 min on a T4.

That number is published in the paper, so reproducing it validates decoding,
normalization, word-level language ID, and both metrics end to end. It is **not**
our baseline — it is the calibration of the measuring instrument. A broken metric
makes every downstream result uninterpretable.

Then decode M6/M7/M8 and the `whisper-small` zero-shot baseline they are measured against.

### Decoding reproduces WhisperX, not a per-clip loop
The paper decodes whole recordings with WhisperX. We use **faster-whisper**, the engine
WhisperX wraps: language detected **once per recording**, 30-second chunks with real
context, beam 5, VAD, temperature fallback, and a compression-ratio threshold that aborts
repetition loops.

This matters enormously. Decoding the 3,136 isolated 6-second clips instead makes Whisper
render English loanwords in Devanagari, so the hypothesis contains **no script boundary and
no switch bigram can match**. Measured on real test audio with one model:

| decoding | %Latin in hyp | MER | CBA-HE |
|---|---|---|---|
| reference | 21.6% | – | – |
| per-clip | **0.0%** | 182.6 | 0.0 |
| recording-level | 12.2% | 103.1 | 1.6 |

`utt_id` is `<speaker>_<recording>_<index>`, so the `test` config already on the Hub is
regrouped into its 30 recordings here — no re-upload.

In [1]:
!pip install -q -U transformers jiwer faster-whisper ctranslate2
!pip install -q "datasets<4" librosa soundfile soxr omegaconf rich

# csasr LAST and FORCED: `pip install git+...` treats an already-installed
# version as satisfied and skips the reinstall, so a re-run in a live kernel
# silently keeps OLD code. --no-deps because the deps are installed above.
!pip install -q --force-reinstall --no-deps git+https://github.com/BRUH-MAIN/codeswitching.git

import csasr
assert csasr.__version__ >= "0.2.0", (
    f"stale csasr {csasr.__version__} - the kernel is running old code. "
    "Restart the kernel (Run > Restart & clear) and re-run this cell."
)
print("csasr", csasr.__version__)

import os, subprocess, sys
os.environ["HF_HOME"] = "/kaggle/temp/hf"

from kaggle_secrets import UserSecretsClient
# Put the token in the ENVIRONMENT, never in argv: a CLI arg lands verbatim in
# every traceback and in `ps` output.
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import login; login(token=os.environ["HF_TOKEN"])

REAL = "RohanRamesh/mucs-he-cs"
OUT  = "/kaggle/working"

def run(*args):
    print(">", " ".join(str(a) for a in args), flush=True)
    subprocess.run([sys.executable, "-m", *args], check=True)   # inherits HF_TOKEN

import torch
N_GPU = torch.cuda.device_count()
print(f"{N_GPU} GPU(s) visible")
!nvidia-smi --query-gpu=name,memory.total --format=csv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 96.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 51.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 56.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 79.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 78.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 8.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


2 GPU(s) visible
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


## Decoding: use **both** T4s and batch the chunks

Sequential decoding pinned one GPU at ~4 GB of 15 GB and left the second idle — large-v2
took 1h50m. Two independent fixes:

* **Batched inference** (`--batch-size 16`). VAD carves the recording into speech chunks and
  they are decoded as a batch instead of one at a time: **8× faster**, measured. This is not a
  shortcut — batched VAD inference is exactly what WhisperX does, so it is *more* faithful to
  the paper than our sequential loop was.
* **Shard recordings across the two GPUs**, one process each: another **2×**.

Together: large-v2 ≈ **1h50m → ~8 min**.

`--lang-detect-segments 8` also lands here. faster-whisper detects the language from a *single*
30-second window by default, and one bad window sends a whole recording into Urdu — where no
Hindi/English switch bigram can match and CBA collapses. Voting over 8 windows fixes it.

In [2]:
from pathlib import Path
from csasr.manifest import read_jsonl, write_jsonl
from csasr.eval.ct2 import resolve_ct2

CT2 = "/kaggle/temp/ct2"

def decode(model, out_name, batch_size=16):
    """Shard the 30 recordings across every GPU, decode in parallel, merge."""
    # Convert ONCE up front: two processes racing on the same cache dir would
    # corrupt it. Prebuilt OpenAI models pass straight through.
    resolve_ct2(model, cache_dir=CT2, quantization="float16")

    n = max(1, N_GPU)
    procs = []
    for i in range(n):
        cmd = [sys.executable, "-m", "csasr.eval.decode",
               "--model", model, "--engine", "faster-whisper", "--mode", "recording",
               "--language", "none", "--batch-size", str(batch_size),
               "--lang-detect-segments", "8",
               "--test-hf", REAL, "--test-config", "test", "--ct2-cache", CT2,
               "--shard", str(i), "--num-shards", str(n),
               "--out", f"{OUT}/{out_name}.shard{i}.jsonl",
               "--refs-out", f"{OUT}/refs.shard{i}.jsonl"]
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i))
        print(f"> GPU{i}: {model} shard {i}/{n}", flush=True)
        procs.append(subprocess.Popen(cmd, env=env))

    for i, p in enumerate(procs):
        if p.wait() != 0:
            raise RuntimeError(f"decode shard {i} failed (exit {p.returncode})")

    hyps = [r for i in range(n) for r in read_jsonl(f"{OUT}/{out_name}.shard{i}.jsonl")]
    refs = [r for i in range(n) for r in read_jsonl(f"{OUT}/refs.shard{i}.jsonl")]
    write_jsonl(f"{OUT}/{out_name}.jsonl", hyps)
    write_jsonl(f"{OUT}/refs_recording.jsonl", sorted(refs, key=lambda r: r["utt_id"]))
    print(f"merged {len(hyps)} recordings -> {OUT}/{out_name}.jsonl")
    return hyps

## GATE 3 — calibrate the metric against a published number

Inference only, ~8 min on 2× T4.

In [3]:
decode("openai/whisper-large-v2", "hyp_largev2_zeroshot")

> GPU0: openai/whisper-large-v2 shard 0/2
> GPU1: openai/whisper-large-v2 shard 1/2


Generating train split: 100%|██████████| 3136/3136 [00:01<00:00, 1807.77 examples/s]


[decode] shard 1/2: 15 recording(s), 1,616 segments
[decode] faster-whisper Systran/faster-whisper-large-v2 on cuda[0] (float16)
[decode] batched pipeline, batch_size=16
[decode] 15 recording(s), language=None (auto-detect per recording, voting over 8 windows)
[decode] shard 0/2: 15 recording(s), 1,520 segments
[decode] faster-whisper Systran/faster-whisper-large-v2 on cuda[0] (float16)
[decode] batched pipeline, batch_size=16
[decode] 15 recording(s), language=None (auto-detect per recording, voting over 8 windows)


decode: 100%|██████████| 15/15 [10:06<00:00, 40.46s/rec]


[decode] wrote 15 hypotheses -> /kaggle/working/hyp_largev2_zeroshot.shard1.jsonl
[decode] wrote 15 recording-level references -> /kaggle/working/refs.shard1.jsonl

  score on CPU:
    python -m csasr.eval.score --refs /kaggle/working/refs.shard1.jsonl --hyps /kaggle/working/hyp_largev2_zeroshot.shard1.jsonl


decode: 100%|██████████| 15/15 [11:35<00:00, 46.36s/rec]


[decode] wrote 15 hypotheses -> /kaggle/working/hyp_largev2_zeroshot.shard0.jsonl
[decode] wrote 15 recording-level references -> /kaggle/working/refs.shard0.jsonl

  score on CPU:
    python -m csasr.eval.score --refs /kaggle/working/refs.shard0.jsonl --hyps /kaggle/working/hyp_largev2_zeroshot.shard0.jsonl
merged 30 recordings -> /kaggle/working/hyp_largev2_zeroshot.jsonl


[{'utt_id': '406yMKxIdSDHRf8H',
  'hyp': 'LibreOffice Impress पर स्लाइड मास्टर और स्लाइड डिजाइन के लिए इस spoken tutorial में आपका स्वागत है। इस tutorial में हम सीखेंगे की स्लाइड्स के लिए background, layout कैसे लागू करें। यहाँ हम अपने operating system के रूप में GNU, Linux और LibreOffice Suite version 3.3.4 का उप्योग कर रहे हैं। Background, स्लाइड पर लागू LibreOffice Impress में कई Background Options हैं जो बहतर प्रस्तुतियां बनाने में आपकी मदद करते हैं। आप भी अपने खुद के Custom Backgrounds बना सकते हैं। SampleImpress.odp प्रस्तुति खोले। अपनी प्रस्तुति के लिए Custom Background बनाएं। हम इस Background को प्रस्तुति की सभी Slides में भी लागू करेंगे। हम इस Background को ब प्रस्तुति के सभी स्लाइड्स के लिए लागू होता है। में मैन्यू से व्यू पर क्लिक करें, मास्टर चुनें और स्लाइड मास्टर पर क्लिक करें। मास्टर स्लाइड प्रदर्शित होती है। ध्यान दें कि मास्टर व्यू टूल्बार भी दिखाई देता है। आप इस जिनका इस प्रस्तुति में उप्योख किया गया है। Tasks Pan में Master Pages पर क्लिक करें। Field इस प्रस्तुति में

In [4]:
from csasr.eval.score import score

for mode in ("word", "hybrid"):
    res = score(f"{OUT}/refs_recording.jsonl", f"{OUT}/hyp_largev2_zeroshot.jsonl", mer_mode=mode)
    print(f"MER mode={mode:<7} -> MER {res['mer']:.1f}   CBA-HE {res['cba_he']:.1f}   CBA-EH {res['cba_eh']:.1f}")

print("\npaper (large-v2 zero-shot): MER 52.0   CBA-HE 42.9   CBA-EH 36.x")
print("Whichever mode lands near 52.0 is the definition the authors used.")
print("\nIf MER is far from 52 or CBA-HE far from 42.9, STOP. Do not generate data")
print("against a ruler that does not reproduce a published number.")

MER mode=word    -> MER 54.8   CBA-HE 20.4   CBA-EH 17.6
MER mode=hybrid  -> MER 51.9   CBA-HE 20.4   CBA-EH 17.6

paper (large-v2 zero-shot): MER 52.0   CBA-HE 42.9   CBA-EH 36.x
Whichever mode lands near 52.0 is the definition the authors used.

If MER is far from 52 or CBA-HE far from 42.9, STOP. Do not generate data
against a ruler that does not reproduce a published number.


### Diagnostics — read these before trusting the numbers above

In [5]:
from collections import Counter
from csasr.manifest import read_jsonl
from csasr.eval.cba import cba
from csasr.eval.mer import mer
from csasr.lid import Lang, count_words
from csasr.normalize import normalize

refs = {r["utt_id"]: r["text"] for r in read_jsonl(f"{OUT}/refs_recording.jsonl")}
hyps = list(read_jsonl(f"{OUT}/hyp_largev2_zeroshot.jsonl"))
R = [refs[h["utt_id"]] for h in hyps]
H = [h["hyp"] for h in hyps]

def mix(t):
    c = count_words(normalize(t, "scoring")); tot = sum(c.values()) or 1
    return c[Lang.HI] / tot, c[Lang.EN] / tot

# 1) BOTH scripts must be present, or CBA is structurally zero regardless of MER.
rh, re_ = mix(" ".join(R)); hh, he = mix(" ".join(H))
print(f"REFERENCE : {rh:5.1%} Devanagari  {re_:5.1%} Latin")
print(f"HYPOTHESIS: {hh:5.1%} Devanagari  {he:5.1%} Latin")

# 2) Hindi/Urdu confusion destroys switch points wholesale.
print("\ndetected language per recording:", dict(Counter(h["detected_language"] for h in hyps)))
def wc(t): return sum(count_words(normalize(t, "scoring")).values())
bad = [h for h in hyps if h["detected_language"] != "hi"]
if bad:
    share = sum(wc(refs[h["utt_id"]]) for h in bad) / sum(wc(t) for t in R)
    print(f"  non-hi: {len(bad)}/{len(hyps)} recordings = {share:.1%} of reference words")
    hi = [h for h in hyps if h["detected_language"] == "hi"]
    ch = cba([refs[h["utt_id"]] for h in hi], [h["hyp"] for h in hi])
    ca = cba(R, H)
    print(f"  CBA-HE  all {ca.he:.1f}  ->  hi-only {ch.he:.1f}")
    print(f"  CBA-EH  all {ca.eh:.1f}  ->  hi-only {ch.eh:.1f}")

# 3) Is any residual MER gap definitional rather than a quality gap?
print()
for p in ("raw", "punct", "scoring"):
    print(f"MER preset={p:8}: {mer(R, H, preset=p):.1f}")

REFERENCE : 75.1% Devanagari  24.4% Latin
HYPOTHESIS: 78.9% Devanagari  17.1% Latin

detected language per recording: {'hi': 29, 'ur': 1}
  non-hi: 1/30 recordings = 2.9% of reference words
  CBA-HE  all 20.4  ->  hi-only 21.2
  CBA-EH  all 17.6  ->  hi-only 18.3

MER preset=raw     : 63.2
MER preset=punct   : 59.1
MER preset=scoring : 54.7


## whisper-small zero-shot — the actual baseline for M6/M7/M8

Expect its CBA ≈ 0: whisper-small transliterates English into Devanagari, so it has no script boundary to match. That is the model, not the pipeline — and it is exactly what fine-tuning is supposed to fix.

In [6]:
decode("openai/whisper-small", "hyp_small_zeroshot")

> GPU0: openai/whisper-small shard 0/2
> GPU1: openai/whisper-small shard 1/2
[decode] shard 1/2: 15 recording(s), 1,616 segments
[decode] faster-whisper Systran/faster-whisper-small on cuda[0] (float16)
[decode] batched pipeline, batch_size=16
[decode] 15 recording(s), language=None (auto-detect per recording, voting over 8 windows)
[decode] shard 0/2: 15 recording(s), 1,520 segments
[decode] faster-whisper Systran/faster-whisper-small on cuda[0] (float16)
[decode] batched pipeline, batch_size=16
[decode] 15 recording(s), language=None (auto-detect per recording, voting over 8 windows)


decode: 100%|██████████| 15/15 [03:15<00:00, 13.04s/rec]


[decode] wrote 15 hypotheses -> /kaggle/working/hyp_small_zeroshot.shard1.jsonl
[decode] wrote 15 recording-level references -> /kaggle/working/refs.shard1.jsonl

  score on CPU:
    python -m csasr.eval.score --refs /kaggle/working/refs.shard1.jsonl --hyps /kaggle/working/hyp_small_zeroshot.shard1.jsonl


decode: 100%|██████████| 15/15 [03:35<00:00, 14.36s/rec]


[decode] wrote 15 hypotheses -> /kaggle/working/hyp_small_zeroshot.shard0.jsonl
[decode] wrote 15 recording-level references -> /kaggle/working/refs.shard0.jsonl

  score on CPU:
    python -m csasr.eval.score --refs /kaggle/working/refs.shard0.jsonl --hyps /kaggle/working/hyp_small_zeroshot.shard0.jsonl
merged 30 recordings -> /kaggle/working/hyp_small_zeroshot.jsonl


[{'utt_id': '406yMKxIdSDHRf8H',
  'hyp': 'लिबर अफिस इंप्रस पर स्लाइद मास्तर अर स्लाइद दिशान के लिए स्पोकें टिट्योल में अपका स्वागत है. इस टिट्योल में हम सीखेंगे की स्लाइद के लिए बैक्ग्रूंँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँँ लिबर अफिस इंप्रेस में कई बैग्राउन अप्षन्स हैं जो बहतर प्रस्तुतिया बनाने में आपकी मदध करते हैं आप भी अपने खुटके कस्तम बैग्राउन्स बना सकते हैं सैंपल इंप्रस डोड अदीपी प्रस्तुती खोले अपनी प्रस्तुती के लिए कस्तम बै प्रस्तुती के सबही लागु होता है में मैन्यू से वूब पर क्लिक करें मास्तर चुनें और स्लाइड मास्तर पर क्लिक करें मास्तर स्लाइड प्रदर्षित होती है द्यान दें की मास्तर वूँ तुल्बार भी दिखाई देता है आप इसका उप्योग मास्तर पेजेज तयार जिनका इस प्रस्तूटी में उप्योख क्या गया है तास्क्स पैन में मास्तर पेजेस पर क्लिक करें फिल्ड इस प्रस्तूटी में इस प्रस्तूटी में इस्तिमाल क्ये मास्तर स्लाइट्स पर अचित करता है मास्तर स्लाइट तेम्प्लेट जैसी है आप यहां फोरमैटिंट और पेज पर क्लिख करें पेज सेटर डलोबोक्स परशिथ होता हैं बैग्राउन टैप पर क्लिख करें फिल ड्रोब्डाउन मेन्यू से बिट मैप अप्षन चुनें

## Decode the fine-tuned models

`decode.py` converts each HF checkpoint to CTranslate2 on first use and caches it, so
faster-whisper can load it. Run this **only after** `02_train.ipynb` has pushed the repos —
otherwise you get a 404, which is expected, not a bug.

In [7]:
for m in ("m6", "m7", "m8"):
    decode(f"RohanRamesh/whisper-small-cs-{m}", f"hyp_{m}")

[ct2] converting RohanRamesh/whisper-small-cs-m6 -> /kaggle/temp/ct2/RohanRamesh__whisper-small-cs-m6 (float16)


OSError: RohanRamesh/whisper-small-cs-m6 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

## Results — reproduce the ordering of Table 2

In [ ]:
systems = [
    ("large-v2 zero-shot", "hyp_largev2_zeroshot.jsonl", 52.0),
    ("small zero-shot",    "hyp_small_zeroshot.jsonl",   None),
    ("M6 (T1, 8h)",        "hyp_m6.jsonl",               48.2),
    ("M7 (T2, 22h)",       "hyp_m7.jsonl",               40.8),
    ("M8 (T2 + mono)",     "hyp_m8.jsonl",               39.2),
]
rows = []
for name, f, paper_mer in systems:
    r = score(f"{OUT}/refs_recording.jsonl", f"{OUT}/{f}")
    rows.append((name, r, paper_mer))

print(f"{'system':<20}{'MER':>8}{'paper':>8}{'CBA-HE':>9}{'CBA-EH':>9}")
for name, r, pm in rows:
    p = f"{pm:.1f}" if pm else "-"
    print(f"{name:<20}{r['mer']:>8.1f}{p:>8}{r['cba_he']:>9.1f}{r['cba_eh']:>9.1f}")

mers = [r["mer"] for _, r, _ in rows[2:]]
print("\nM6 > M7 > M8 ordering reproduced:", mers == sorted(mers, reverse=True))

with open(f"{OUT}/table2.json", "w") as fh:
    json.dump([{"system": n, **r} for n, r, _ in rows], fh, indent=2)